# Section 5: Evaluation

*Duration: 15 minutes*

---

The agent loop ran. It selected tools, rewrote queries, and produced answers. But did it actually improve things?

This section scores the agent loop on two dimensions, not one. Answer correctness measures whether the final output matches the expected answer. Reasoning correctness measures whether the agent used the right tool for the right reason. A correct answer reached through wrong reasoning is luck, not reliability. A wrong answer reached through correct reasoning is a corpus gap, not an architecture failure.

Separating these two dimensions is the difference between knowing your system works and hoping it works.

## 5.1 Loading Results

Three result sets tell the full story:

1. **Passive RAG baseline** — the Escalation Lab's final evaluation (eval_results.json)
2. **Agent loop results** — the tool-augmented loop from Section 4 (agent_loop_results.json)

In [1]:
import json

# Load the passive RAG baseline from the Escalation Lab
with open("../prebuilt/eval_results.json", "r", encoding="utf-8") as f:
    baseline_data = json.load(f)

baseline_results = baseline_data["results"]

# Load agent loop results from Section 4
with open("../prebuilt/agent_loop_results.json", "r", encoding="utf-8") as f:
    agent_data = json.load(f)

agent_results = agent_data["results"]

print(f"Baseline questions : {len(baseline_results)}")
print(f"Agent loop results : {len(agent_results)}")

Baseline questions : 10
Agent loop results : 10


## 5.2 Side-by-Side Comparison

The table below shows every question, the passive RAG result, and the agent loop result. Look for three patterns:

- **Preserved passes** — questions the passive pipeline got right that the agent loop also got right (no regression)
- **Recovered failures** — questions the passive pipeline got wrong that the agent loop fixed
- **Persistent failures** — questions neither architecture could answer correctly

In [2]:
# Build a side-by-side comparison table
# The delta column highlights what changed between passive and agent

print(f"{'ID':<6} {'Category':<22} {'Passive':<12} {'Agent':<10} {'Delta'}")
print("=" * 75)

passive_pass_count = 0
agent_pass_count = 0
recovered = 0
regressed = 0

for br in baseline_results:
    # Find matching agent result
    ar = next((a for a in agent_results if a["id"] == br["id"]), None)
    
    passive_ok = br["classification"] == "pass"
    agent_ok = ar["agent_classification"] == "pass" if ar else False
    
    if passive_ok:
        passive_pass_count += 1
    if agent_ok:
        agent_pass_count += 1
    
    # Determine what changed
    if not passive_ok and agent_ok:
        delta = "RECOVERED"
        recovered += 1
    elif passive_ok and not agent_ok:
        delta = "REGRESSED"
        regressed += 1
    elif passive_ok and agent_ok:
        delta = "\u2014"
    else:
        delta = "still failing"
    
    passive_display = "pass" if passive_ok else "FAIL"
    agent_display = "pass" if agent_ok else "FAIL"
    
    print(f"{br['id']:<6} {br.get('category', ''):<22} {passive_display:<12} {agent_display:<10} {delta}")

print("=" * 75)
print(f"{'Total':<6} {'':<22} {passive_pass_count}/10{'':<7} {agent_pass_count}/10")
print(f"\nRecovered: {recovered}  |  Regressed: {regressed}  |  Net change: +{recovered - regressed}")

ID     Category               Passive      Agent      Delta
q01    explicit_rule          pass         pass       —
q02    terminology            FAIL         pass       RECOVERED
q03    implicit_reasoning     FAIL         pass       RECOVERED
q04    table_lookup           pass         pass       —
q05    multi_step_rule        pass         pass       —
q06    table_lookup           pass         FAIL       REGRESSED
q07    terminology            pass         pass       —
q08    implicit_reasoning     pass         pass       —
q09    explicit_rule          pass         pass       —
q10    implicit_reasoning     pass         pass       —
Total                         8/10        9/10

Recovered: 2  |  Regressed: 1  |  Net change: +1


## 5.3 Two-Dimensional Scoring

Answer correctness is necessary but not sufficient. A system that produces the right answer through the wrong reasoning path is unreliable — it will break on the next similar question where luck does not hold.

For each agent loop result, score on two dimensions:

- **answer_correct** — does the final answer match the expected answer?
- **reasoning_correct** — did the agent select the right tool for the right reason?

The reasoning dimension requires human judgment. Look at the trace for each question and assess: was the tool selection appropriate? Were the query arguments reasonable? Did the agent use the tool result correctly?

> **Facilitator note:** This is a discussion exercise. Walk through 2-3 examples as a group. The point is not to score every question perfectly — it is to demonstrate that answer correctness alone is an incomplete evaluation.

In [3]:
# Two-dimensional scoring
# answer_correct comes from the model judge in Section 4
# reasoning_correct is filled in by participants based on trace inspection
#
# Pre-filled with reasonable defaults based on tool selection patterns.
# Participants should review and adjust based on the traces.

scoring = []

for ar in agent_results:
    answer_correct = ar.get("agent_classification") == "pass"
    
    # Default reasoning assessment based on tool selection pattern
    # Participants should override these after inspecting traces
    tools_used = ar.get("tools_used", [])
    
    # A question that uses rag_retrieval for a rules question is reasoning-correct
    # A question that uses no_answer when corpus lacks info is reasoning-correct
    # A question that uses calculator for a math question is reasoning-correct
    reasoning_correct = len(tools_used) > 0  # default: correct if any tool was used
    
    scoring.append({
        "id": ar["id"],
        "question": ar["question"][:60],
        "answer_correct": answer_correct,
        "reasoning_correct": reasoning_correct,
        "tools_used": tools_used
    })

# Display the scoring table
print(f"{'ID':<6} {'Answer':<10} {'Reasoning':<12} {'Tools Used':<30} {'Question'}")
print("=" * 100)
for s in scoring:
    ans = "correct" if s["answer_correct"] else "WRONG"
    rsn = "correct" if s["reasoning_correct"] else "WRONG"
    tools = ", ".join(s["tools_used"]) if s["tools_used"] else "none"
    print(f"{s['id']:<6} {ans:<10} {rsn:<12} {tools:<30} {s['question']}")

ID     Answer     Reasoning    Tools Used                     Question
q01    correct    correct      rag_retrieval                  What happens if a Thief fails an Open Locks attempt?
q02    correct    correct      rag_retrieval                  Why can't Elves roll higher than a d6 for hit points?
q03    correct    correct      rag_retrieval                  Can a character wear leather armor and cast spells?
q04    correct    correct      rag_retrieval                  What is the saving throw for a 3rd level Fighter against Dra
q05    correct    correct      rag_retrieval                  How does a Cleric turn undead?
q06    WRONG      correct      calculator                     If a character has a Strength of 16, what bonus do they get 
q07    correct    correct      rag_retrieval                  What is the difference between a retainer and a hireling?
q08    correct    correct      rag_retrieval                  When can a Magic-User learn new spells?
q09    correct    corre

## 5.4 The 2x2 Matrix

The two scoring dimensions — answer correctness and reasoning correctness — produce four quadrants. Each quadrant tells you something different about the system, and each one points to a different kind of intervention.

**Reliable** (correct answer, correct reasoning) is the only quadrant you want to be in. The agent selected the right tool, passed reasonable arguments, used the result appropriately, and produced a correct answer. This is a system you can trust to handle similar questions tomorrow.

**Lucky** (correct answer, wrong reasoning) is the most dangerous quadrant. The final answer happens to be correct, but the path that produced it was wrong — the model selected the wrong tool, or used the right tool with poor arguments, and still landed on the right answer by coincidence. A simple accuracy metric cannot distinguish Lucky from Reliable. But the next similar question will expose the difference, because luck does not generalize.

**Corpus Gap** (wrong answer, correct reasoning) is the most actionable quadrant. The agent did everything right — selected the right tool, passed a good query, evaluated the results — but the corpus did not contain the information needed to answer. The fix is upstream: improve the data, not the agent. This is a content problem, not an architecture problem.

**Tool Problem** (wrong answer, wrong reasoning) means the agent went off the rails. The tool selection was wrong, the arguments were wrong, or both. Start by examining the trace and the tool descriptions. This is typically a description-writing problem, not a model problem.

In [4]:
# Categorize each result into the 2x2 matrix
# Each quadrant tells you something different about system reliability

reliable = []        # correct answer + correct reasoning
lucky = []           # correct answer + wrong reasoning
corpus_gap = []      # wrong answer + correct reasoning
tool_problem = []    # wrong answer + wrong reasoning

for s in scoring:
    if s["answer_correct"] and s["reasoning_correct"]:
        reliable.append(s["id"])
    elif s["answer_correct"] and not s["reasoning_correct"]:
        lucky.append(s["id"])
    elif not s["answer_correct"] and s["reasoning_correct"]:
        corpus_gap.append(s["id"])
    else:
        tool_problem.append(s["id"])

print("2x2 Evaluation Matrix")
print("=" * 60)
print()
print(f"  Correct answer + Correct reasoning : RELIABLE")
print(f"    {reliable if reliable else '(none)'}")
print()
print(f"  Correct answer + Wrong reasoning   : LUCKY (not reliable)")
print(f"    {lucky if lucky else '(none)'}")
print()
print(f"  Wrong answer + Correct reasoning   : CORPUS GAP (fixable)")
print(f"    {corpus_gap if corpus_gap else '(none)'}")
print()
print(f"  Wrong answer + Wrong reasoning     : TOOL/DEFINITION PROBLEM")
print(f"    {tool_problem if tool_problem else '(none)'}")
print()
print("=" * 60)
print(f"Reliable: {len(reliable)}/10  |  Lucky: {len(lucky)}/10  |  "
      f"Corpus gap: {len(corpus_gap)}/10  |  Tool problem: {len(tool_problem)}/10")

2x2 Evaluation Matrix

  Correct answer + Correct reasoning : RELIABLE
    ['q01', 'q02', 'q03', 'q04', 'q05', 'q07', 'q08', 'q09', 'q10']

  Correct answer + Wrong reasoning   : LUCKY (not reliable)
    (none)

  Wrong answer + Correct reasoning   : CORPUS GAP (fixable)
    ['q06']

  Wrong answer + Wrong reasoning     : TOOL/DEFINITION PROBLEM
    (none)

Reliable: 9/10  |  Lucky: 0/10  |  Corpus gap: 1/10  |  Tool problem: 0/10


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 7), facecolor="#0f172a")
ax.set_facecolor("#0f172a")

# Quadrant definitions: (x, y, color, label, description, items)
quadrants = [
    (0, 1, "#22c55e", "RELIABLE",
     "Correct answer\nCorrect reasoning",
     reliable),
    (1, 1, "#f59e0b", "LUCKY",
     "Correct answer\nWrong reasoning",
     lucky),
    (0, 0, "#3b82f6", "CORPUS GAP",
     "Wrong answer\nCorrect reasoning",
     corpus_gap),
    (1, 0, "#ef4444", "TOOL PROBLEM",
     "Wrong answer\nWrong reasoning",
     tool_problem),
]

for x, y, color, label, desc, items in quadrants:
    # Draw quadrant background
    alpha = 0.25 if items else 0.08
    rect = mpatches.FancyBboxPatch(
        (x + 0.03, y + 0.03), 0.94, 0.94,
        boxstyle="round,pad=0.02",
        facecolor=color, alpha=alpha,
        edgecolor=color, linewidth=2
    )
    ax.add_patch(rect)

    # Quadrant label
    ax.text(x + 0.5, y + 0.82, label,
            ha="center", va="center", fontsize=16,
            fontweight="bold", color=color)

    # Description
    ax.text(x + 0.5, y + 0.65, desc,
            ha="center", va="center", fontsize=9,
            color="#94a3b8", linespacing=1.4)

    # Count
    ax.text(x + 0.5, y + 0.45, f"{len(items)}/10",
            ha="center", va="center", fontsize=28,
            fontweight="bold", color="white")

    # Question IDs
    if items:
        ids_text = ", ".join(items)
    else:
        ids_text = "(none)"
    ax.text(x + 0.5, y + 0.2, ids_text,
            ha="center", va="center", fontsize=9,
            color="white", alpha=0.7,
            style="italic")

# Axis labels
ax.text(1.0, -0.08, "Reasoning Correctness \u2192",
        ha="center", va="center", fontsize=11,
        color="#94a3b8", fontweight="bold")
ax.text(1.0, -0.15, "correct                                           wrong",
        ha="center", va="center", fontsize=9, color="#64748b")

ax.text(-0.12, 1.0, "Answer\nCorrectness",
        ha="center", va="center", fontsize=11,
        color="#94a3b8", fontweight="bold", rotation=90)
ax.text(-0.06, 1.5, "correct", ha="center", va="center",
        fontsize=9, color="#64748b", rotation=90)
ax.text(-0.06, 0.5, "wrong", ha="center", va="center",
        fontsize=9, color="#64748b", rotation=90)

# Dividing lines
ax.axhline(y=1, xmin=0.0, xmax=1.0, color="#334155", linewidth=1.5)
ax.axvline(x=1, ymin=0.0, ymax=1.0, color="#334155", linewidth=1.5)

ax.set_xlim(-0.2, 2.1)
ax.set_ylim(-0.25, 2.1)
ax.set_aspect("equal")
ax.axis("off")

ax.set_title("2x2 Evaluation Matrix: Answer Correctness \u00d7 Reasoning Correctness",
             fontsize=14, fontweight="bold", color="white", pad=20)

fig.text(0.5, 0.01,
         "Only the Reliable quadrant represents a trustworthy system. "
         "Lucky results fool accuracy metrics but fail on the next similar question.",
         ha="center", fontsize=9, color="#64748b", style="italic")

plt.tight_layout(rect=[0.05, 0.03, 1, 1])
plt.show()

The 2x2 matrix is not an academic exercise. It is the evaluation framework that determines whether an agentic system is ready for production.

A deployment decision based solely on accuracy — "the agent got 9 out of 10 right" — cannot distinguish a reliable system from a lucky one. If three of those nine correct answers came through wrong reasoning, the system will fail unpredictably in production when it encounters similar questions where luck does not hold. The 2x2 matrix surfaces this risk before deployment, not after.

In enterprise contexts, each quadrant maps to a different stakeholder conversation. Reliable results are ready to ship. Lucky results need tool description work before shipping. Corpus gaps need a conversation with the data team, not the engineering team. Tool problems need trace inspection and possibly architecture changes. The matrix tells you not just *whether* to intervene, but *who* needs to intervene and *where* in the stack the fix lives.

> **Facilitator note:** The "lucky" quadrant is the most important one to discuss. A correct answer from wrong reasoning will fool a simple accuracy metric. It will not fool a user who asks a similar question tomorrow and gets a wrong answer. Reliability requires both dimensions to be correct.

---

## 5.5 Failure Analysis

For any question that is not in the "reliable" quadrant, the 2x2 matrix tells you where to look:

- **Lucky** — The tool description needs refinement. The model selected a tool that happened to produce useful context, but for the wrong reason. Fix the description.
- **Corpus gap** — The agent's reasoning was sound, but the corpus does not contain the answer. Fix the data, not the agent.
- **Tool problem** — Both the tool selection and the answer were wrong. Start by examining the trace: did the model misunderstand the tool description, or did the tool return unhelpful results?

Each failure maps to a specific layer. That is the value of two-dimensional scoring: it tells you *where* to intervene, not just *whether* to intervene.

In [5]:
# For each non-reliable result, print the diagnosis
print("Failure Analysis")
print("=" * 60)

for s in scoring:
    if s["answer_correct"] and s["reasoning_correct"]:
        continue  # Skip reliable results
    
    # Find the full agent result for trace details
    ar = next(a for a in agent_results if a["id"] == s["id"])
    
    if s["answer_correct"] and not s["reasoning_correct"]:
        category = "LUCKY \u2014 correct answer, wrong reasoning"
        action = "Review and fix tool description"
    elif not s["answer_correct"] and s["reasoning_correct"]:
        category = "CORPUS GAP \u2014 wrong answer, correct reasoning"
        action = "Improve corpus coverage or chunking"
    else:
        category = "TOOL PROBLEM \u2014 wrong answer, wrong reasoning"
        action = "Examine trace and fix tool definition or dispatch"
    
    print(f"\n  {s['id']}: {category}")
    print(f"  Question : {ar['question']}")
    print(f"  Tools    : {', '.join(ar.get('tools_used', []))}")
    print(f"  Action   : {action}")
    if ar.get("judge_reason"):
        print(f"  Judge    : {ar['judge_reason']}")

Failure Analysis

  q06: CORPUS GAP — wrong answer, correct reasoning
  Question : If a character has a Strength of 16, what bonus do they get on melee attack rolls?
  Tools    : calculator
  Action   : Improve corpus coverage or chunking
  Judge    : The actual answer is incorrect as a Strength score of 16 provides a +2 bonus, not 3.


## 5.6 Regression Analysis

The side-by-side comparison in 5.2 may show one or more regressions — questions the passive pipeline answered correctly that the agent loop got wrong. A regression means the added complexity made something worse. That is exactly what the escalation principle warns against: complexity must justify itself with evidence, and a regression is evidence against.

The cell below isolates any regressed questions and inspects the agent trace to identify the cause. Regressions in agentic systems typically fall into one of three categories:

- **Tool selection error** — the model chose the wrong tool because the description was too broad. A question about a numeric value in a table is a retrieval problem, not a calculation problem, but a calculator description that mentions "numeric computation" can attract it.
- **Retrieval degradation** — the agent rewrote the query in a way that produced worse chunks than the original query would have.
- **Over-reasoning** — the model saw tool results and second-guessed a correct intuition, producing a worse answer than it would have without tools.

Each cause points to a different fix. Tool selection errors are description problems. Retrieval degradation is a rewriting problem. Over-reasoning is a prompt or control structure problem.

In [ ]:
# Identify and analyze regressions
regressions = []

for br in baseline_results:
    ar = next((a for a in agent_results if a["id"] == br["id"]), None)
    if not ar:
        continue
    passive_ok = br["classification"] == "pass"
    agent_ok = ar.get("agent_classification") == "pass"
    if passive_ok and not agent_ok:
        regressions.append((br, ar))

if not regressions:
    print("No regressions detected. Every question the passive pipeline")
    print("answered correctly, the agent loop also answered correctly.")
else:
    print(f"Found {len(regressions)} regression(s)")
    print("=" * 70)

    for br, ar in regressions:
        print(f"\n  ID       : {br['id']}")
        print(f"  Question : {br['question']}")
        print(f"  Expected : {br['expected']}")
        print(f"  Category : {br.get('category', '')}")
        print(f"  Passive  : pass")
        print(f"  Agent    : FAIL")
        print(f"  Tools    : {', '.join(ar.get('tools_used', []))}")
        print(f"  Answer   : {ar.get('agent_answer', '')[:200]}")

        # Diagnosis based on tools used
        tools = ar.get("tools_used", [])
        category = br.get("category", "")

        if "calculator" in tools and category in ("table_lookup", "explicit_rule", "terminology", "implicit_reasoning"):
            print(f"\n  Diagnosis: TOOL SELECTION ERROR")
            print(f"  The model selected 'calculator' for a {category} question.")
            print(f"  This is a description problem: the calculator description")
            print(f"  mentions 'numeric computation', which attracted a question")
            print(f"  about a numeric value that lives in a table, not a formula.")
            print(f"  Fix: tighten the calculator description to exclude table lookups.")
        elif "no_answer" in tools:
            print(f"\n  Diagnosis: FALSE ABSTENTION")
            print(f"  The agent declined to answer a question the corpus can answer.")
            print(f"  Fix: review retrieval quality or no_answer description.")
        else:
            print(f"\n  Diagnosis: Inspect trace for details.")

        # Show trace steps
        print(f"\n  Trace:")
        for step in ar.get("trace", []):
            print(f"    Step {step.get('iteration', '?')}: {step['tool']}({json.dumps(step.get('arguments', {}))})")
            result = step.get('result', {})
            if 'chunks' in result:
                print(f"      Retrieved {len(result['chunks'])} chunks, top distance: {result['chunks'][0].get('distance', '?')}")
            elif 'result' in result:
                print(f"      Result: {result['result']}")
            elif 'error' in result:
                print(f"      Error: {result['error']}")
        print()

Regressions are the cost of added complexity. The passive pipeline had no tool selection step, so it could not select the wrong tool. The agent loop introduced a decision point, and that decision point can go wrong.

This is not an argument against the agent loop. It is an argument for evaluating it on both dimensions. If you only measured answer accuracy, a regression from 8/10 to 9/10 with one new failure looks like a net win. The 2x2 matrix reveals that the new failure is a tool description problem with a specific, fixable cause. The right response is not to remove the agent loop. It is to fix the description and re-run.

## 5.7 Cumulative Results

The table below shows results across all evaluation approaches: passive RAG baseline from the Escalation Lab and the agent loop from this lab.

In [6]:
# Build the cumulative comparison table
print("Cumulative Evaluation Results")
print("=" * 70)
print(f"{'ID':<6} {'Category':<22} {'Passive RAG':<14} {'Agent Loop'}")
print("-" * 70)

for br in baseline_results:
    ar = next((a for a in agent_results if a["id"] == br["id"]), None)
    
    passive = "pass" if br["classification"] == "pass" else "FAIL"
    agent = "pass" if (ar and ar.get("agent_classification") == "pass") else "FAIL"
    
    print(f"{br['id']:<6} {br.get('category', ''):<22} {passive:<14} {agent}")

passive_total = sum(1 for r in baseline_results if r["classification"] == "pass")
agent_total = sum(1 for r in agent_results if r.get("agent_classification") == "pass")

print("-" * 70)
print(f"{'Total':<6} {'':<22} {passive_total}/10{'':<9} {agent_total}/10")
print()
print(f"Net improvement from agent loop: +{agent_total - passive_total} questions")

Cumulative Evaluation Results
ID     Category               Passive RAG    Agent Loop
----------------------------------------------------------------------
q01    explicit_rule          pass           pass
q02    terminology            FAIL           pass
q03    implicit_reasoning     FAIL           pass
q04    table_lookup           pass           pass
q05    multi_step_rule        pass           pass
q06    table_lookup           pass           FAIL
q07    terminology            pass           pass
q08    implicit_reasoning     pass           pass
q09    explicit_rule          pass           pass
q10    implicit_reasoning     pass           pass
----------------------------------------------------------------------
Total                         8/10          9/10

Net improvement from agent loop: +1 questions


> **FIELD TAKEAWAY**
>
> Accuracy alone is a dangerous metric for agentic systems. A system that gets the right answer through the wrong reasoning path is unreliable — it will fail unpredictably on similar questions. Two-dimensional evaluation (answer correctness + reasoning correctness) separates reliable results from lucky ones, and tells you exactly which layer to fix when something goes wrong: the tool description, the corpus, or the control structure.

---

## What Comes Next

Section 6 is a facilitated discussion. No new code. No live API calls. The results are in. The question now is: what do they mean for real engagements, and when is the added complexity of an agent loop justified?

Move to `06_Synthesis/06_Synthesis.ipynb`.